# Simulating the Amplitude-Based Collisionless QLBM with parallel boundary conditions

A marker register lets one simulation carry several *geometries* at once: each marker state
selects a different set of solid objects, and the measured grid populations are the average
over those geometries.

`qlbm` builds its circuits as `qarp` blocks and simulates them with `qarp`'s statevector simulator.
A run is described by a `SimulationConfig`, which ties the four algorithmic circuits — initial
conditions, time step algorithm, postprocessing, and measurement — to a compiler optimization level
and a shot budget. `cfg.validate()` type-checks the configuration and `cfg.prepare_for_simulation()`
lowers every component into a built `qarp` block. A `QarpRunner` then drives the time loop and writes
one ParaView frame per time step into `output_dir/paraview`.

`statevector_snapshots=True` is the fast path: the runner hands the state produced at step $k$ back to
the simulator as the *initial state* of step $k+1$, so simulating $n$ steps costs $n$ applications of
the algorithm instead of $\frac{n(n+1)}{2}$. Passing `qarp.EXACT` instead of a shot count returns exact
outcome probabilities rather than sampled counts.

> **Demo-sized.** The lattice below is a 16x16 D2Q9 domain carrying two geometries,
> simulated for 10 time steps. Simulation cost grows exponentially in the number of qubits, so scale the lattice and
> `NUM_STEPS` up deliberately.

In [ ]:
from qlbm.components import (
    ABGridMeasurement,
    EmptyPrimitive,
)
from qlbm.components.ab.ab import ABQLBM
from qlbm.components.ab.initial import ABParallelDiscreteUniformInitialConditions
from qlbm.infra import QarpRunner, SimulationConfig
from qlbm.lattice import ABLattice
from qlbm.tools.utils import create_directory_and_parents

In [ ]:
lattice = ABLattice(
    {
        "lattice": {"dim": {"x": 16, "y": 16}, "velocities": "d2q9"},
        "geometry": [],
    }
)

output_dir = "qlbm-output/ab-pbc-d2q9-16x16-qarp"
create_directory_and_parents(output_dir)

In [ ]:
# Geometry of marker |0>: two "stacked" "wide" rectangles
# Geometry of the marker |1>: only the "top" rectangle is present
# These geometries will act fully in parallel, and we will observe the average behavior
lattice.set_geometries(
    [
        [
            {"shape": "cuboid", "x": [6, 12], "y": [12, 14], "boundary": "bounceback"},
            {"shape": "cuboid", "x": [6, 12], "y": [8, 10], "boundary": "bounceback"},
        ],
        [
            {"shape": "cuboid", "x": [6, 12], "y": [12, 14], "boundary": "bounceback"},
        ],
    ]
)

In [ ]:
lattice.geometries

In [ ]:
# Number of shots to sample for each time step
NUM_SHOTS = 2**12

# Number of time steps to simulate
NUM_STEPS = 10

In [ ]:
cfg = SimulationConfig(
    initial_conditions=ABParallelDiscreteUniformInitialConditions(
        lattice,
        [[5], [5]],
        [([0, 1], [0, 1, 2, 3]), ([0, 1], [0, 1, 2, 3])],
    ),
    algorithm=ABQLBM(lattice, use_agnostic_bcs=False),
    postprocessing=EmptyPrimitive(lattice),
    measurement=ABGridMeasurement(lattice),
    optimization_level=0,
    shots=NUM_SHOTS,
)

cfg.validate()
cfg.prepare_for_simulation()

In [ ]:
runner = QarpRunner(cfg, lattice, seed=1234)


# Snapshots carry the statevector between time steps, making the loop O(NUM_STEPS)
runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)